In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import (
    cosine_similarity,
    euclidean_distances)

load_dotenv()

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b:free"

print(EMBEDDING_MODEL)


nvidia/nemotron-3-embed-1b:free


In [2]:
documents = [
    {"id": 1,  "text": "Spring Boot simplifies building production-ready Java backend applications.",                                               "topic": "backend",           "level": "beginner"},
    {"id": 2,  "text": "Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.",                          "topic": "devops",            "level": "intermediate"},
    {"id": 3,  "text": "Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.",                                  "topic": "databases",         "level": "beginner"},
    {"id": 4,  "text": "Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.",                      "topic": "distributed-systems","level": "intermediate"},
    {"id": 5,  "text": "PostgreSQL is a relational database with strong SQL support, transactions, indexing, and extensibility.",                  "topic": "databases",         "level": "beginner"},
    {"id": 6,  "text": "Vector databases store embeddings and make similarity search over high-dimensional vectors efficient.",                    "topic": "ai",                "level": "beginner"},
    {"id": 7,  "text": "RAG retrieves relevant external context and gives that context to a language model before generating an answer.",          "topic": "ai",                "level": "beginner"},
    {"id": 8,  "text": "An embedding model converts text into a dense numerical vector that captures semantic relationships.",                    "topic": "ai",                "level": "beginner"},
    {"id": 9,  "text": "Cosine similarity compares the angle between vectors and is commonly used for semantic text retrieval.",                  "topic": "ai",                "level": "intermediate"},
    {"id": 10, "text": "HNSW builds a navigable graph so approximate nearest-neighbor search can avoid scanning every vector.",                   "topic": "vector-search",     "level": "advanced"},
    {"id": 11, "text": "Exact nearest-neighbor search compares the query against every candidate vector to obtain the true top results.",         "topic": "vector-search",     "level": "intermediate"},
    {"id": 12, "text": "Quantization compresses vector representations so a vector database can reduce memory usage and improve search efficiency.","topic": "vector-search",     "level": "advanced"},
    {"id": 13, "text": "Payload indexes allow a vector database to combine semantic similarity with structured filters.",                         "topic": "vector-search",     "level": "advanced"},
    {"id": 14, "text": "Python is widely used for machine learning, data processing, and building AI applications.",                              "topic": "programming",       "level": "beginner"},
    {"id": 15, "text": "Async programming can improve throughput for applications that spend significant time waiting on network I/O.",           "topic": "programming",       "level": "intermediate"},
    {"id": 16, "text": "API gateways can provide routing, authentication, rate limiting, and observability for microservices.",                   "topic": "distributed-systems","level": "intermediate"},
    {"id": 17, "text": "Microservices split an application into independently deployable services that communicate over APIs or events.",         "topic": "distributed-systems","level": "intermediate"},
    {"id": 18, "text": "Observability combines logs, metrics, and traces to understand the behavior of distributed applications.",               "topic": "devops",            "level": "intermediate"},
    {"id": 19, "text": "A cache can reduce database load and latency when frequently requested data can be reused safely.",                       "topic": "databases",         "level": "beginner"},
    {"id": 20, "text": "Reranking can improve retrieval quality by applying a more expensive relevance model to an initial candidate set.",       "topic": "ai",                "level": "advanced"},
]

texts = [d["text"] for d in documents]
print(f"{len(documents)} documents")
print(texts[0])

20 documents
Spring Boot simplifies building production-ready Java backend applications.


In [3]:
def embed_documents(texts: list[str]) -> list[list[float]]:
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [d.embedding for d in resp.data]

doc_embeddings = embed_documents(texts)

print("Number of embeddings:", len(doc_embeddings))
print("Dimensions per vector:", len(doc_embeddings[0]))
print("First 5 values:", doc_embeddings[0])

Number of embeddings: 20
Dimensions per vector: 2048
First 5 values: [0.09486013650894165, 0.07493621855974197, -0.00033464969601482153, -0.0032216955441981554, 0.01421054732054472, -0.010696923360228539, 0.021084513515233994, -0.03974340856075287, 0.015385974198579788, -0.0048223575577139854, 0.008787916973233223, -0.006052966695278883, 0.0022460296750068665, -0.007704062853008509, 0.010700482875108719, 0.0009034750401042402, 0.0008514579967595637, 0.005959315691143274, 0.03242502361536026, 0.009030004031956196, 0.00038098046206869185, 0.015569517388939857, -0.0179350096732378, 0.005743435584008694, 0.021483244374394417, -0.005791892297565937, -0.009824302047491074, -0.027099506929516792, 0.006302964873611927, 0.003879721276462078, -0.020878028124570847, 0.0054513090290129185, -0.034183319658041, -0.014470435678958893, -0.0060340287163853645, -0.02435862086713314, -0.017685208469629288, 0.00208463822491467, -0.014235270209610462, 0.0033797244541347027, 0.018665621057152748, 0.02187327

In [4]:
query = "Which programming language is commonly used for AI applications?"

query_embedding = embed_documents([query])   # returns a list with one vector

print("Query:", query)
print("Dimensions:", len(query_embedding[0]))
print("Collection must use size =", len(doc_embeddings[0]))

Query: Which programming language is commonly used for AI applications?
Dimensions: 2048
Collection must use size = 2048


| Metric | Intuition | Better match |
|---|---|---|
| Cosine | Angle between vectors | Higher |
| Euclidean / L2 | Straight-line distance | Lower |
| Dot product | Alignment + magnitude | Higher |

In [5]:
cos = cosine_similarity(query_embedding, doc_embeddings)[0]
l2  = euclidean_distances(query_embedding, doc_embeddings)[0]

print(f"{'Cos':>6}  {'L2':>5}  Document")
print("-" * 70)
for doc, c, d in zip(documents, cos, l2):
    print(f"{c:6.3f}  {d:6.3f}  {doc}")

   Cos     L2  Document
----------------------------------------------------------------------
 0.116   1.329  {'id': 1, 'text': 'Spring Boot simplifies building production-ready Java backend applications.', 'topic': 'backend', 'level': 'beginner'}
 0.078   1.358  {'id': 2, 'text': 'Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.', 'topic': 'devops', 'level': 'intermediate'}
 0.152   1.302  {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}
 0.100   1.341  {'id': 4, 'text': 'Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.', 'topic': 'distributed-systems', 'level': 'intermediate'}
 0.058   1.373  {'id': 5, 'text': 'PostgreSQL is a relational database with strong SQL support, transactions, indexing, and extensibility.', 'topic': 'databases', 'level': 'beginner'}
 0.061   1.370  {'id': 6, '

In [23]:
def retrieve(query_emb, doc_embs, docs, top_k=2):
    scores = cosine_similarity(query_emb, doc_embs)[0]

    # pair each document with its score, sort highest first, take top_k
    scored = list(zip(docs, scores))
    # print(scored)
    scored.sort(key=lambda x: x[1], reverse=True)

    return [{"document": doc, "score": float(score)} for doc, score in scored[:top_k]]


print("Top-K results:\n")
for hit in retrieve(query_embedding, doc_embeddings, documents, top_k=4):
    print(f"{hit['score']:.3f}  {hit['document']}")

Top-K results:

0.614  {'id': 14, 'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'}
0.199  {'id': 15, 'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'}
0.152  {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}
0.137  {'id': 7, 'text': 'RAG retrieves relevant external context and gives that context to a language model before generating an answer.', 'topic': 'ai', 'level': 'beginner'}


---
## Qdrant Vector Database

In [20]:
# %pip install qdrant-client
from qdrant_client import QdrantClient, models

# qdrant = QdrantClient(":memory:")

qdrant = QdrantClient(url="http://localhost:6333")

In [ ]:
# del qdrant

In [21]:
COLLECTION = "rag_fundamentals"

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size=len(doc_embeddings[0]),   # always read from the model, never hard-code
        distance=models.Distance.COSINE,
    ),
)

print("Created:", COLLECTION, "| size:", len(doc_embeddings[0]))

Created: rag_fundamentals | size: 2048


In [22]:
points = [
    models.PointStruct(
        id=doc["id"],
        vector=vec,
        payload={"text": doc["text"], "topic": doc["topic"], "level": doc["level"]},
    )
    for doc, vec in zip(documents, doc_embeddings)
]

qdrant.upsert(collection_name=COLLECTION, points=points, wait=True)
print(f"Inserted {len(points)} points.")

Inserted 20 points.


In [24]:
results = qdrant.query_points(
    collection_name=COLLECTION,
    query=query_embedding[0],    # a single vector, not a list
    limit=2,
    with_payload=True,
).points

for p in results:
    print(p)

id=14 version=1 score=0.61424613 payload={'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'} vector=None shard_key=None order_value=None
id=15 version=1 score=0.19856398 payload={'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'} vector=None shard_key=None order_value=None


In [28]:
compressedClient = qdrant.create_collection(
    collection_name="compressed2",
    vectors_config=models.VectorParams(size=len(doc_embeddings[0]), distance=models.Distance.COSINE),
    quantization_config=models.ScalarQuantization(
        scalar=models.ScalarQuantizationConfig(
            type=models.ScalarType.INT8,   # float32 -> 8-bit integers
            always_ram=True,
        )
    ),
)

In [29]:
qdrant.upsert(collection_name="compressed2", points=points, wait=True)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [30]:
results = qdrant.query_points(
    collection_name="compressed2",
    query=query_embedding[0],
    limit=2,
    with_payload=True,
    with_vectors=True,
).points

print("First 2 values of vector:", results[0].vector[:2])

First 2 values of vector: [0.07419223, 0.025613999]
